Step 1- Import everything we need

In [98]:
import os
from dotenv import load_dotenv
#Langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [99]:
load_dotenv()

True

In [100]:
groq_key= os.getenv("GROQ_API_KEY")
jina_key= os.getenv("JINA_API_KEY")

In [101]:
DATA_FILE_PATH=os.path.join("data", "hr_policy.txt")

DATA INGESTION

In [102]:
loader = TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents= loader.load()

### LANGCHAIN DOCUMENT
Langchain processes everythin in form in documents
1. PAGE CONTENT: the actutal data/content
2. METADATA: extra information about the data

In [103]:
print(f"Loaded {len(documents)} documents")

Loaded 1 documents


In [104]:
print(f"Metadata of first document: {documents[0].metadata}")

Metadata of first document: {'source': 'data\\hr_policy.txt'}


In [105]:
print(f"total characters in first document: {len(documents[0].page_content)}")

total characters in first document: 2597


SPLIITING OUR DATA

In [106]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks= text_splitter.split_documents(documents)
print(chunks)
print(f"Loaded {len(chunks)} chunks")

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

NOW EACH SPLITTED CHUNK IS DOCUMENT- page content and metadata

In [107]:
print(f"content of first chunk: {chunks[0].page_content}")

content of first chunk: COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)


In [108]:
print(f"{chunks[8].page_content}")

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


EMBED OUR DATA

In [109]:
embeddings_model= JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("EMBEDDINGS MODEL: ", embeddings_model.model_name)

EMBEDDINGS MODEL:  jina-embeddings-v2-base-en


STORE DATA IN VECTOR DB

In [110]:
vector_store= FAISS.from_documents(chunks, embeddings_model)
print("CHUNKS ARE STORRED IN VECTOR STORE", vector_store.index.ntotal)

CHUNKS ARE STORRED IN VECTOR STORE 9


In [111]:
test_query= "How many sick leave employees get?"
## SIMILARITY SEARCH
similar_docs= vector_store.similarity_search(test_query, k=3)
print(f"Query: {test_query}\n")
for i, doc in enumerate(similar_docs):
    print(f"Document {i+1}: {doc.page_content}\n")

Query: How many sick leave employees get?

Document 1: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

Document 2: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

Document 3: 3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Per

In [112]:
### TOOL


In [113]:
retriever= vector_store.as_retriever(search_type="similarity", search_kwargs={"k":3})
def search_hr_policy(query: str)->str:
    """
    Search the HR policy documents for relevant information based on the query. For the information about leave, probation period, salary, and other HR policies, this function will return the relevant information from the documents.

    """
    matching_chunks= retriever.invoke(query)
    return "\n\n".join([chunk.page_content for chunk in matching_chunks])

DATA RETRIEVAL- LLM

In [114]:
llm= ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.5) # temperature means creativity
llm.model_name

'openai/gpt-oss-120b'

In [115]:
test_response= llm.invoke("Is learing RAG hard? Answer in 1 line. Reoly in a funny manner. In one line")
print(f"Response: {test_response.content}")

Response: Learning RAG is about as easy as teaching a cat to fetch—possible, but expect a lot of confused meowing and occasional hairballs!


LLM= BRAIN, TOOL- SUPER POWER, MEMORY- No Memory

In [116]:
hr_assistant= create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR Assistant working for ACME Corp. Always use the search_hr_policy tool
    to lookup facts before answering, 
    If the asnwer isn't in the search results, 
    say you don't know "insted of guessing"
    """
)

print("HR Assistant is ready to answer questions!")

HR Assistant is ready to answer questions!


In [117]:
def ask_hr_assistant(question: str)-> str:
    """ Send a question to the HR Assistant and return the answer """
    print("="*60)
    print("QUESTION: ", question)
    print("-"*60)
    response= hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer= response.messages[-1].content
    print("ANSWER: ", answer)
    print("="*60)
    print()
    return answer


In [118]:
response= hr_assistant.invoke({"messages": [{"role": "user", "content": "Tell me about leave policies. How to apply for leave?"}]})

In [119]:
print(response["messages"][-1].content)

**ACME Corp – Leave Policies & How to Apply**

---

### 1. Types of Leave

| Leave Type | Entitlement | Key Rules |
|------------|-------------|-----------|
| **Annual (Paid) Leave** | 20 days per calendar year | • Must be requested **at least 5 working days** before the start date.<br>• Up to **5 unused days** can be carried forward to the next year. |
| **Sick Leave** | 10 paid days per year | • A medical certificate is required **if the sick leave is longer than 2 consecutive days**.<br>• Sick leave is **separate** from annual leave. |
| **Public Holidays** | 12 days (as per the annual holiday calendar) | • If you work on a public holiday you are eligible for **compensatory leave**. |

---

### 2. How to Apply for Leave

1. **Log in to the HR Portal**  
   - URL: *[HR portal link]* (accessible via the intranet or your desktop shortcuts).

2. **Navigate to the “Leave Request” Section**  
   - Click **“New Request”**.

3. **Select the Leave Type**  
   - Choose **Annual Leave**, **Sic